# Feature Extraction and Machine Learning

## Overview

You will turn each trial into EEG power spectral density (PSD) features and build a logistic regression classifier. The notebook supplies the evaluation structure but leaves model construction, feature sets, and interpretation code for you to complete.

## Learning objectives

- Build a feature matrix `X` and aligned label vector `y`.
- Place scaling inside a scikit-learn pipeline to avoid leakage.
- Compare an 80/20 split with stratified 5-fold cross-validation.
- Interpret a confusion matrix and standardized coefficients.
- Compare physiologically motivated feature sets fairly.


## Background: Machine learning goal

The model predicts whether one 15-second trial came from eyes-open or eyes-closed EEG. Chance is near 50% when classes are balanced. This is within-recording classification, not evidence that the model generalizes to new people. A useful result should agree with plausible physiology and remain stable across held-out folds.

## What are EEG features?

A feature is a numerical trial summary given to a model. Here, columns represent summaries of the alpha-band PSD from channels or regions. Different features can have different numerical ranges, so we will later use `StandardScaler` inside the machine-learning pipeline. Keeping the scaler inside the pipeline ensures that its means and standard deviations are learned from training data only.

In [ ]:
# Setup cell: run this as provided.
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import (
    StratifiedKFold, cross_val_score, train_test_split
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.eeg_utils import plot_confusion_matrix_from_predictions

RANDOM_STATE = 42
mne.set_log_level("WARNING")
print(f"Using MNE version {mne.__version__}")


## Step 1: Load epochs

Load the saved epochs with `mne.read_epochs(..., preload=True)`. Print the event dictionary and condition counts. Verify that both task conditions exist and that their trial counts match what you retained in Notebook 3.

In [ ]:
EPOCHS_PATH = Path("../outputs/eyes_open_closed_epo.fif")

# TODO: load epochs into memory.
epochs = ...

# TODO: verify Start Eyes Open and Start Eyes Closed are present, then print counts.
required_conditions = {"Start Eyes Open", "Start Eyes Closed"}
...


## Step 2: Extract alpha PSD features from each channel

Alpha frequencies in the brain are usually assumed to be between the range of 8-12 Hz. Call `epochs.compute_psd()` and use welch's method to compute the PSD within this range. Calling `.get_data()` on the result gives an array shaped **trials × channels × frequency bins**.

To build our machine-learning feature matrix, we will simplify the full frequency spectrum by calculating the mean alpha power per trial and channel. Calculate the mean alpha power over the final frequency axis by using `axis=-1`.

Create a DataFrame to visualize your mean alpha features for each trial and channel.

In [ ]:
# TODO: compute the 8–12 Hz PSD and inspect the returned array shape.
alpha_spectrum = epochs.compute_psd(...)
alpha_psd = ...
print("PSD shape (trials, channels, frequencies):", ...)

# TODO: average only frequency bins, leaving one mean value per trial and EEG channel.
alpha_mean_psd = ...

# TODO: make a labeled DataFrame for the mean alpha features for each channel.
mean_feature_names = [f"alpha_mean_psd_{channel}" for channel in epochs.ch_names]
alpha_mean = pd.DataFrame(...)


## Step 3: Build feature matrix X and label vector y

MNE already stores an integer event code for every epoch. The third column of `epochs.events` contains these codes.

Print `epochs.event_id` to see what each integer means—for example, `Eyes Open: 1` and `Eyes Closed: 2`. Rows in `X` remain aligned with rows in `epochs.events`. Finally, print the feature-matrix shape and class counts and confirm that the numbers of rows and labels match.

In [ ]:
# TODO: print the condition-to-integer mapping stored by MNE.
print(...)

# TODO: build the feature matrix X as a copy of alpha_mean
X = ...

# TODO: build the label vector y by extracting the third column from epochs.events
y = ...

# TODO: print dimensions and class counts
print("Feature matrix shape:", ...)
print("Event-code counts", pd.Series(...).value_counts())

## Step 4: First-pass logistic regression with an 80/20 split

Use `train_test_split` with `test_size=0.20`, `stratify=y`, and the provided random state. Build a `Pipeline` containing `StandardScaler` followed by `LogisticRegression(max_iter=1000)`. Fit only with training data, predict the test labels, and calculate accuracy.

**Leakage check:** You should never call `scaler.fit_transform(X)` before splitting. The pipeline learns scaling parameters only from the training set.

In [ ]:
# TODO: make a stratified 80/20 split.
X_train, X_test, y_train, y_test = train_test_split(
    ..., ...,
    test_size=...,
    stratify=...,
    random_state=RANDOM_STATE,
)

# TODO: construct the two-step pipeline.
model = Pipeline(
    steps=[
        ("scaler", ...),
        ("logistic_regression", ...),
    ]
)

# TODO: fit on training data, predict test data, and calculate accuracy.
...

## Step 5: Plot a confusion matrix

Call `plot_confusion_matrix_from_predictions` with the true and predicted test labels. Sort `epochs.event_id` by event code so the displayed condition names line up with the integer labels. Report the raw counts: with a small test set, one trial changes the result substantially.

In [ ]:
# Sort the MNE mapping by event code to keep labels and names aligned.
sorted_events = sorted(epochs.event_id.items(), key=lambda item: item[1])
encoded_labels = [event_code for _, event_code in sorted_events]
display_names = [name.replace("_", " " ).title() for name, _ in sorted_events]

# TODO: create the confusion-matrix display and replace numeric tick labels.
display_matrix = plot_confusion_matrix_from_predictions(
    ..., ..., labels=encoded_labels
)
display_matrix.ax_.set_xticklabels(display_names)
display_matrix.ax_.set_yticklabels(display_names)
plt.show()

## Step 6: Evaluate with 5-fold cross-validation

One split can be unusually easy or difficult. Create `StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)`. Pass the complete pipeline—not a pre-scaled matrix—to `cross_val_score`. Print every fold plus mean and sample standard deviation. Five-fold stratification requires at least five trials in each class.

In [ ]:
# TODO: check that the smaller class contains at least five trials.
...

# TODO: configure stratified five-fold CV and evaluate accuracy.
cv = StratifiedKFold(
    n_splits=..., shuffle=..., random_state=...
)
cv_scores = cross_val_score(..., ..., ..., cv=..., scoring=...)

# TODO: print fold scores and mean ± sample SD (ddof=1).
...

## Step 7: Compare feature sets

The goal is to compare feature sets without rebuilding a new DataFrame by hand every time. The helper functions below do three common jobs:

1. Compute one PSD statistic for one frequency band and return a labeled feature table.
2. Select only a set of named channels, such as posterior channels.
3. Concatenate multiple feature tables, such as posterior alpha plus posterior beta.

Use the exact same `model`, `cv`, and `y` for every feature set. The model pipeline still contains `StandardScaler`, so the features can stay in their original PSD units. More columns are not automatically better, especially with a small number of trials.


In [ ]:
def make_band_features(epochs, band_name, fmin, fmax, statistic="mean"):
    """Return one PSD statistic per trial and channel for a frequency band."""
    spectrum = epochs.compute_psd(
        method="welch", fmin=fmin, fmax=fmax, picks="eeg", verbose=False
    )
    psd = spectrum.get_data()  # trials × channels × frequency bins

    if statistic == "mean":
        values = psd.mean(axis=-1)
    elif statistic == "std":
        values = psd.std(axis=-1)
    elif statistic == "max":
        values = psd.max(axis=-1)
    else:
        raise ValueError("statistic must be 'mean', 'std', or 'max'")

    columns = [f"{band_name}_{statistic}_psd_{ch}" for ch in spectrum.ch_names]
    return pd.DataFrame(values, columns=columns)


def available_channels(epochs, candidate_names):
    """Keep candidate channel names that are actually present in this recording."""
    candidates = {name.casefold() for name in candidate_names}
    return [ch for ch in epochs.ch_names if ch.casefold() in candidates]


def pick_channel_features(feature_df, channel_names):
    """Select feature columns that end with one of the requested channel names."""
    selected_columns = [
        column
        for channel in channel_names
        for column in feature_df.columns
        if column.endswith(f"_{channel}")
    ]
    return feature_df.loc[:, selected_columns].copy()


def average_channel_features(feature_df, new_column_name, channel_names):
    """Average selected channel columns into one regional feature column."""
    selected = pick_channel_features(feature_df, channel_names)
    if selected.empty:
        return None
    return pd.DataFrame({new_column_name: selected.mean(axis=1)})


def combine_feature_blocks(*feature_dfs):
    """Concatenate feature tables column-wise while keeping trial rows aligned."""
    return pd.concat(feature_dfs, axis=1)


In [ ]:
alpha_mean_all = make_band_features(epochs, "alpha", 8, 12, statistic="mean")
alpha_std_all = make_band_features(epochs, "alpha", 8, 12, statistic="std")
alpha_max_all = make_band_features(epochs, "alpha", 8, 12, statistic="max")
theta_mean_all = make_band_features(epochs, "theta", 4, 8, statistic="mean")
beta_mean_all = make_band_features(epochs, "beta", 13, 30, statistic="mean")

feature_sets = {
    "Alpha mean PSD: all channels": alpha_mean_all,
    "Alpha PSD standard deviation: all channels": alpha_std_all,
    "Alpha maximum PSD: all channels": alpha_max_all,
    "Theta + alpha + beta mean PSD: all channels": combine_feature_blocks(
        theta_mean_all, alpha_mean_all, beta_mean_all
    ),
}

posterior_candidates = {
    "P1", "P2", "P3", "P4", "P5", "P6", "P7", "P8", "Pz",
    "PO3", "PO4", "PO7", "PO8", "POz", "O1", "O2", "Oz",
}
posterior_channels = available_channels(epochs, posterior_candidates)
print("Posterior channels found:", posterior_channels if posterior_channels else "none")

if posterior_channels:
    posterior_alpha = pick_channel_features(alpha_mean_all, posterior_channels)
    posterior_beta = pick_channel_features(beta_mean_all, posterior_channels)

    feature_sets["Alpha mean PSD: posterior channels"] = posterior_alpha
    feature_sets["Posterior alpha + beta mean PSD"] = combine_feature_blocks(
        posterior_alpha, posterior_beta
    )

region_candidates = {
    "frontal": {"Fp1", "Fp2", "Fpz", "F3", "F4", "F7", "F8", "Fz"},
    "central": {"C3", "C4", "Cz", "FC1", "FC2", "CP1", "CP2"},
    "posterior": posterior_candidates,
}
regional_blocks = []
for region_name, candidates in region_candidates.items():
    region_channels = available_channels(epochs, candidates)
    region_feature = average_channel_features(
        alpha_mean_all, f"alpha_mean_psd_{region_name}", region_channels
    )
    if region_feature is not None:
        regional_blocks.append(region_feature)

regional_blocks.append(pd.DataFrame({"alpha_mean_psd_global": alpha_mean_all.mean(axis=1)}))
feature_sets["Alpha mean PSD: regional averages"] = combine_feature_blocks(*regional_blocks)

print("Feature sets ready to compare:")
for feature_set_name, feature_matrix in feature_sets.items():
    print(f"- {feature_set_name}: {feature_matrix.shape[1]} columns")


In [ ]:
comparison_rows = []
for feature_set_name, feature_matrix in feature_sets.items():
    # Reusing cv applies identical splitting logic; cross_val_score refits each fold.
    scores = cross_val_score(model, feature_matrix, y, cv=cv, scoring="accuracy")
    comparison_rows.append(
        {
            "feature_set": feature_set_name,
            "n_features": feature_matrix.shape[1],
            "mean_accuracy": scores.mean(),
            "sd_accuracy": scores.std(ddof=1),
        }
    )

comparison = pd.DataFrame(comparison_rows).sort_values("mean_accuracy", ascending=False)
display(comparison.style.format({"mean_accuracy": "{:.3f}", "sd_accuracy": "{:.3f}"}))


## Step 8: Interpret model coefficients

For interpretation only, fit a fresh all-channel alpha pipeline on all trials. This fit must not be reported as held-out performance. Extract `coef_[0]` from the fitted logistic-regression pipeline step. Plot one bar per channel. Because the inputs were standardized, magnitudes are comparable. For binary logistic regression, positive values favor the second value in the fitted model's `classes_` array. Use the reverse of `epochs.event_id` to translate that integer back to a condition name. Correlated channels can redistribute weights, so coefficients are not independent physiological effects.


In [ ]:
# TODO: build a fresh scaler/logistic pipeline and fit all alpha trials.
coefficient_model = ...
...

# TODO: retrieve coefficients, pair them with channel names, and sort.
coefficients = ...
coefficient_series = ...

# TODO: translate the model's positive event code back to a condition name.
positive_class_name = ...
...

## Step 9: Discussion and limitations

Any learned transformation, feature selection, artifact threshold, or hyperparameter tuning must occur inside training folds. Repeatedly changing features to maximize the same CV score also creates optimism; use nested CV or a separate final test set for a tuned workflow.

Trials from one person do not demonstrate generalization to new people. A multi-participant analysis should keep every participant's trials in one fold using grouped splitting. Condition blocks can also be confounded with drift, fatigue, or time. Interpret high accuracy alongside PSDs, topographies, trial counts, class balance, artifacts, and timing.


## Student exercises

1. Compare the pipeline with and without `StandardScaler`. Keep both versions inside a pipeline.
2. Plot every fold score for every feature set instead of only the mean.
3. Sliding-window challenge: use `epochs.get_data(picks="eeg")` to get the raw epoch array shaped trials × channels × time. Choose a window length and step size in samples, make overlapping windows within each trial, reshape the windows so each row is one training example, and expand the labels so they match the new rows. For example, if each trial becomes `n_windows` windows ordered trial-by-trial, `np.repeat(y, n_windows)` gives one label per window.
4. Inspect spectra from misclassified trials without deleting trials merely because the model was wrong.


## Reflection questions

- Did the classifier perform above chance, and how variable were folds?
- Which features were useful, and did that agree with posterior alpha physiology?
- What did the confusion matrix show that accuracy did not?
- What could cause misleadingly high accuracy?
- If you make overlapping windows, why are those rows not fully independent new trials?
- What data and splitting method are needed to claim generalization to new people?


## Summary

After completing the TODOs, you will have implemented trial-level feature extraction, leakage-safe logistic regression, held-out and cross-validated evaluation, feature-set comparison, and cautious coefficient interpretation.
